In [1]:
# VERSION 2 - STRONGER CATBOOST PIPELINE

import pandas as pd
import numpy as np
import pygeohash as pgh

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# ==========================================
# LOAD DATA
# ==========================================

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

# ==========================================
# TIME FEATURES
# ==========================================

train[['hour', 'minute']] = (
    train['timestamp']
    .str.split(':', expand=True)
    .astype(int)
)

test[['hour', 'minute']] = (
    test['timestamp']
    .str.split(':', expand=True)
    .astype(int)
)

# cyclic hour
train["hour_sin"] = np.sin(2*np.pi*train["hour"]/24)
train["hour_cos"] = np.cos(2*np.pi*train["hour"]/24)

test["hour_sin"] = np.sin(2*np.pi*test["hour"]/24)
test["hour_cos"] = np.cos(2*np.pi*test["hour"]/24)

# ==========================================
# TRAFFIC PERIOD FEATURES
# ==========================================

for df in [train, test]:
    df["is_morning_rush"] = (
        (df["hour"] >= 7) &
        (df["hour"] <= 9)
    ).astype(int)

    df["is_evening_rush"] = (
        (df["hour"] >= 17) &
        (df["hour"] <= 19)
    ).astype(int)

    df["is_peak"] = (
        df["is_morning_rush"] |
        df["is_evening_rush"]
    ).astype(int)

    df["time_slot"] = (
        df["hour"] * 4 +
        (df["minute"] // 15)
    )

# ==========================================
# MISSING VALUES
# ==========================================

train["RoadType"] = train["RoadType"].fillna("Unknown")
test["RoadType"] = test["RoadType"].fillna("Unknown")

train["Weather"] = train["Weather"].fillna("Unknown")
test["Weather"] = test["Weather"].fillna("Unknown")

temp_median = train["Temperature"].median()

train["Temperature"] = train["Temperature"].fillna(temp_median)
test["Temperature"] = test["Temperature"].fillna(temp_median)

# ==========================================
# GEOHASH FEATURES
# ==========================================

def decode_geohash(g):
    lat, lon = pgh.decode(g)
    return pd.Series([float(lat), float(lon)])

train[["lat", "lon"]] = train["geohash"].apply(decode_geohash)
test[["lat", "lon"]] = test["geohash"].apply(decode_geohash)

# geohash hierarchy
train["geo4"] = train["geohash"].str[:4]
train["geo5"] = train["geohash"].str[:5]

test["geo4"] = test["geohash"].str[:4]
test["geo5"] = test["geohash"].str[:5]

# geohash + hour
train["geo_hour"] = (
    train["geohash"] + "_" +
    train["hour"].astype(str)
)

test["geo_hour"] = (
    test["geohash"] + "_" +
    test["hour"].astype(str)
)

# ==========================================
# SIMPLE TARGET AGGREGATES
# ==========================================

geo_mean = (
    train.groupby("geohash")["demand"]
    .mean()
    .to_dict()
)

train["geo_mean_demand"] = (
    train["geohash"]
    .map(geo_mean)
)

test["geo_mean_demand"] = (
    test["geohash"]
    .map(geo_mean)
)

global_mean = train["demand"].mean()

test["geo_mean_demand"] = (
    test["geo_mean_demand"]
    .fillna(global_mean)
)

# ==========================================
# REMOVE TIMESTAMP
# ==========================================

train.drop(columns=["timestamp"], inplace=True)
test.drop(columns=["timestamp"], inplace=True)

# ==========================================
# FEATURES
# ==========================================

TARGET = "demand"

FEATURES = [c for c in train.columns if c != TARGET]

X = train[FEATURES]
y = train[TARGET]

X_test = test[FEATURES]

# ==========================================
# CAT FEATURES
# ==========================================

cat_features = [
    "geohash",
    "geo4",
    "geo5",
    "geo_hour",
    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

# ==========================================
# KFOLD
# ==========================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        loss_function="RMSE",
        eval_metric="R2",
        random_seed=42,
        verbose=500
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    preds = model.predict(X_valid)

    score = r2_score(y_valid, preds)

    scores.append(score)

    print(f"Fold {fold} R2 = {score:.5f}")

print("\nMean CV R2 =", np.mean(scores))

# ==========================================
# FINAL MODEL
# ==========================================

final_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=500
)

final_model.fit(X, y, cat_features=cat_features)

# ==========================================
# FEATURE IMPORTANCE
# ==========================================

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": final_model.get_feature_importance()
})

importance = importance.sort_values("Importance", ascending=False)

print(importance.head(20))

# ==========================================
# SUBMISSION
# ==========================================

predictions = final_model.predict(X_test)

predictions = np.clip(predictions, 0, None)

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": predictions
})

submission.to_csv("../submissions/submission.csv", index=False)

print("\nsubmission.csv generated")

Train Shape: (77299, 11)
Test Shape : (41778, 10)
0:	learn: 0.0504881	test: 0.0510432	best: 0.0510432 (0)	total: 147ms	remaining: 7m 21s
500:	learn: 0.9527827	test: 0.9481506	best: 0.9481506 (500)	total: 46.7s	remaining: 3m 53s
1000:	learn: 0.9616149	test: 0.9535605	best: 0.9535605 (1000)	total: 1m 35s	remaining: 3m 11s
1500:	learn: 0.9659403	test: 0.9556366	best: 0.9556387 (1499)	total: 2m 22s	remaining: 2m 22s
2000:	learn: 0.9689376	test: 0.9567891	best: 0.9567899 (1999)	total: 3m 9s	remaining: 1m 34s
2500:	learn: 0.9712928	test: 0.9575503	best: 0.9575536 (2499)	total: 3m 53s	remaining: 46.5s
2999:	learn: 0.9731117	test: 0.9581050	best: 0.9581052 (2998)	total: 4m 38s	remaining: 0us

bestTest = 0.9581051505
bestIteration = 2998

Shrink model to first 2999 iterations.
Fold 1 R2 = 0.95811
0:	learn: 0.0504314	test: 0.0501790	best: 0.0501790 (0)	total: 104ms	remaining: 5m 13s
500:	learn: 0.9524867	test: 0.9490199	best: 0.9490199 (500)	total: 46.9s	remaining: 3m 54s
1000:	learn: 0.9612385	

In [4]:
# ==========================================
# VERSION 3 - TARGET ENCODING FOCUSED
# ==========================================

import pandas as pd
import numpy as np
import pygeohash as pgh

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

# ==========================================
# LOAD DATA
# ==========================================

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

# ==========================================
# TIME FEATURES
# ==========================================

train[['hour', 'minute']] = (
    train['timestamp']
    .str.split(':', expand=True)
    .astype(int)
)

test[['hour', 'minute']] = (
    test['timestamp']
    .str.split(':', expand=True)
    .astype(int)
)

# cyclical encoding

train["hour_sin"] = np.sin(
    2 * np.pi * train["hour"] / 24
)

train["hour_cos"] = np.cos(
    2 * np.pi * train["hour"] / 24
)

test["hour_sin"] = np.sin(
    2 * np.pi * test["hour"] / 24
)

test["hour_cos"] = np.cos(
    2 * np.pi * test["hour"] / 24
)

for df in [train, test]:

    df["is_morning_rush"] = (
        (df["hour"] >= 7) &
        (df["hour"] <= 9)
    ).astype(int)

    df["is_evening_rush"] = (
        (df["hour"] >= 17) &
        (df["hour"] <= 19)
    ).astype(int)

    df["is_peak"] = (
        df["is_morning_rush"] |
        df["is_evening_rush"]
    ).astype(int)

    df["is_night"] = (
        (df["hour"] >= 22) |
        (df["hour"] <= 5)
    ).astype(int)

    df["time_slot"] = (
        df["hour"] * 4 +
        (df["minute"] // 15)
    )

# ==========================================
# MISSING VALUES
# ==========================================

for df in [train, test]:

    df["RoadType"] = (
        df["RoadType"]
        .fillna("Unknown")
    )

    df["Weather"] = (
        df["Weather"]
        .fillna("Unknown")
    )

temp_median = train["Temperature"].median()

train["Temperature"] = (
    train["Temperature"]
    .fillna(temp_median)
)

test["Temperature"] = (
    test["Temperature"]
    .fillna(temp_median)
)

# ==========================================
# GEOHASH FEATURES
# ==========================================

def decode_geohash(g):

    lat, lon = pgh.decode(g)

    return pd.Series([
        float(lat),
        float(lon)
    ])

train[["lat", "lon"]] = (
    train["geohash"]
    .apply(decode_geohash)
)

test[["lat", "lon"]] = (
    test["geohash"]
    .apply(decode_geohash)
)

train["geo3"] = train["geohash"].str[:3]
train["geo4"] = train["geohash"].str[:4]
train["geo5"] = train["geohash"].str[:5]

test["geo3"] = test["geohash"].str[:3]
test["geo4"] = test["geohash"].str[:4]
test["geo5"] = test["geohash"].str[:5]

train["geo_hour"] = (
    train["geohash"] + "_" +
    train["hour"].astype(str)
)

test["geo_hour"] = (
    test["geohash"] + "_" +
    test["hour"].astype(str)
)

# ==========================================
# LEAKAGE SAFE TARGET ENCODING
# ==========================================

global_mean = train["demand"].mean()

kf_te = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ----------------------------------
# geohash mean demand
# ----------------------------------

train["geo_mean_demand"] = np.nan

for tr_idx, val_idx in kf_te.split(train):

    tr = train.iloc[tr_idx]
    val = train.iloc[val_idx]

    geo_map = (
        tr.groupby("geohash")["demand"]
        .mean()
    )

    train.loc[val_idx,
              "geo_mean_demand"] = (
        val["geohash"]
        .map(geo_map)
    )

train["geo_mean_demand"] = (
    train["geo_mean_demand"]
    .fillna(global_mean)
)

geo_map_full = (
    train.groupby("geohash")["demand"]
    .mean()
)

test["geo_mean_demand"] = (
    test["geohash"]
    .map(geo_map_full)
    .fillna(global_mean)
)

# ----------------------------------
# geohash + hour mean demand
# ----------------------------------

train["geo_hour_mean_demand"] = np.nan

for tr_idx, val_idx in kf_te.split(train):

    tr = train.iloc[tr_idx]
    val = train.iloc[val_idx]

    geo_hour_map = (
        tr.groupby(
            ["geohash", "hour"]
        )["demand"]
        .mean()
    )

    train.loc[
        val_idx,
        "geo_hour_mean_demand"
    ] = (
        val.set_index(
            ["geohash", "hour"]
        )
        .index
        .map(geo_hour_map)
    )

train["geo_hour_mean_demand"] = (
    train["geo_hour_mean_demand"]
    .fillna(global_mean)
)

geo_hour_map_full = (
    train.groupby(
        ["geohash", "hour"]
    )["demand"]
    .mean()
)

test["geo_hour_mean_demand"] = (
    test.set_index(
        ["geohash", "hour"]
    )
    .index
    .map(geo_hour_map_full)
)

test["geo_hour_mean_demand"] = (
    test["geo_hour_mean_demand"]
    .fillna(
        test["geo_mean_demand"]
    )
)

# ----------------------------------
# road type mean demand
# ----------------------------------

train["roadtype_mean_demand"] = np.nan

for tr_idx, val_idx in kf_te.split(train):

    tr = train.iloc[tr_idx]
    val = train.iloc[val_idx]

    road_map = (
        tr.groupby("RoadType")["demand"]
        .mean()
    )

    train.loc[
        val_idx,
        "roadtype_mean_demand"
    ] = (
        val["RoadType"]
        .map(road_map)
    )

train["roadtype_mean_demand"] = (
    train["roadtype_mean_demand"]
    .fillna(global_mean)
)

road_map_full = (
    train.groupby("RoadType")["demand"]
    .mean()
)

test["roadtype_mean_demand"] = (
    test["RoadType"]
    .map(road_map_full)
    .fillna(global_mean)
)

# ==========================================
# REMOVE TIMESTAMP
# ==========================================

train.drop(
    columns=["timestamp"],
    inplace=True
)

test.drop(
    columns=["timestamp"],
    inplace=True
)

# ==========================================
# FEATURES
# ==========================================

TARGET = "demand"

FEATURES = [
    c for c in train.columns
    if c != TARGET
]

X = train[FEATURES]
y = train[TARGET]

X_test = test[FEATURES]

# ==========================================
# CAT FEATURES
# ==========================================

cat_features = [

    "geohash",

    "geo3",
    "geo4",
    "geo5",

    "geo_hour",

    "RoadType",
    "LargeVehicles",
    "Landmarks",
    "Weather"
]

# ==========================================
# KFOLD TRAINING
# ==========================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for fold, (
    train_idx,
    val_idx
) in enumerate(
    kf.split(X),
    1
):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[val_idx]

    model = CatBoostRegressor(

        iterations=3000,

        learning_rate=0.03,

        depth=8,

        loss_function="RMSE",

        eval_metric="R2",

        random_seed=42,

        verbose=500
    )

    model.fit(

        X_train,

        y_train,

        cat_features=cat_features,

        eval_set=(
            X_valid,
            y_valid
        ),

        use_best_model=True
    )

    preds = model.predict(
        X_valid
    )

    score = r2_score(
        y_valid,
        preds
    )

    scores.append(score)

    print(
        f"Fold {fold} R2 = {score:.5f}"
    )

print(
    "\nMean CV R2 =",
    np.mean(scores)
)

# ==========================================
# FINAL MODEL
# ==========================================

final_model = CatBoostRegressor(

    iterations=3000,

    learning_rate=0.03,

    depth=8,

    loss_function="RMSE",

    random_seed=42,

    verbose=500
)

final_model.fit(
    X,
    y,
    cat_features=cat_features
)

# ==========================================
# FEATURE IMPORTANCE
# ==========================================

feature_importance = pd.DataFrame({

    "Feature": FEATURES,

    "Importance": final_model.get_feature_importance()

})

feature_importance = (
    feature_importance
    .sort_values(
        by="Importance",
        ascending=False
    )
)

print("\nTOP 20 FEATURES:\n")

print(
    feature_importance.head(20)
)

# optional csv

feature_importance.to_csv(
    "feature_importance.csv",
    index=False
)

# ==========================================
# PREDICTION
# ==========================================

predictions = final_model.predict(
    X_test
)

predictions = np.clip(
    predictions,
    0,
    1
)

submission = pd.DataFrame({

    "Index": test["Index"],

    "demand": predictions
})

submission.to_csv(
    "../submissions/submission.csv",
    index=False
)

print(
    "\nsubmission.csv generated"
)

Train Shape: (77299, 11)
Test Shape : (41778, 10)
0:	learn: 0.0530347	test: 0.0531199	best: 0.0531199 (0)	total: 93.5ms	remaining: 4m 40s
500:	learn: 0.9706093	test: 0.9656399	best: 0.9656399 (500)	total: 52.5s	remaining: 4m 21s
1000:	learn: 0.9775534	test: 0.9701473	best: 0.9701473 (1000)	total: 1m 42s	remaining: 3m 23s
1500:	learn: 0.9807442	test: 0.9717647	best: 0.9717716 (1496)	total: 2m 32s	remaining: 2m 32s
2000:	learn: 0.9829746	test: 0.9727654	best: 0.9727662 (1999)	total: 3m 25s	remaining: 1m 42s
2500:	learn: 0.9846329	test: 0.9734474	best: 0.9734474 (2500)	total: 4m 17s	remaining: 51.3s
2999:	learn: 0.9858721	test: 0.9739500	best: 0.9739522 (2995)	total: 5m 9s	remaining: 0us

bestTest = 0.9739521783
bestIteration = 2995

Shrink model to first 2996 iterations.
Fold 1 R2 = 0.97395
0:	learn: 0.0530322	test: 0.0521956	best: 0.0521956 (0)	total: 90.1ms	remaining: 4m 30s
500:	learn: 0.9699661	test: 0.9616136	best: 0.9616136 (500)	total: 51.1s	remaining: 4m 14s
1000:	learn: 0.976985

In [11]:
import numpy as np
import pandas as pd
import pygeohash as pgh
from catboost import CatBoostRegressor
import lightgbm as lgb
# data lives next to the notebook (falls back to a data/ folder or the uploads path)
import os
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
print('train:', train.shape, ' test:', test.shape)
train.head()
geo_cache = {}
def latlon(g):
    if g not in geo_cache:
        lat, lon = pgh.decode(g)
        geo_cache[g] = (float(lat), float(lon))
    return geo_cache[g]

def add_features(df):
    df = df.copy()
    hm = df['timestamp'].astype(str).str.split(':', expand=True).astype(int)
    df['hour'], df['minute'] = hm[0], hm[1]
    df['tmin'] = df['hour'] * 60 + df['minute']
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['tmin_sin'] = np.sin(2 * np.pi * df['tmin'] / 1440)
    df['tmin_cos'] = np.cos(2 * np.pi * df['tmin'] / 1440)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['is_peak']  = (((df.hour >= 7) & (df.hour <= 9)) | ((df.hour >= 17) & (df.hour <= 19))).astype(int)
    df['is_night'] = ((df.hour < 6) | (df.hour >= 22)).astype(int)
    df['time_slot'] = df['hour'] * 4 + (df['minute'] // 15)
    df['geo5'] = df['geohash'].str[:5]
    df['geo4'] = df['geohash'].str[:4]
    df['geo3'] = df['geohash'].str[:3]
    coords = df['geohash'].map(latlon)
    df['lat'] = coords.map(lambda t: t[0])
    df['lon'] = coords.map(lambda t: t[1])
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    df['Weather']  = df['Weather'].fillna('Unknown')
    return df

train = add_features(train)
test  = add_features(test)

temp_median = train['Temperature'].median()
train['Temperature'] = train['Temperature'].fillna(temp_median)
test['Temperature']  = test['Temperature'].fillna(temp_median)
global_mean = train['demand'].mean()

def smoothed_encoding(col, m):
    stats = train.groupby(col)['demand'].agg(['mean', 'count'])
    enc = (stats['mean'] * stats['count'] + global_mean * m) / (stats['count'] + m)
    return enc

# for col, m in [('geohash', 10), ('geo5', 20), ('geo4', 30)]:
for col, m in [('geohash', 15), ('geo5', 25), ('geo4', 40)]:
    enc = smoothed_encoding(col, m)
    train[col + '_te'] = train[col].map(enc).fillna(global_mean)
    test[col + '_te']  = test[col].map(enc).fillna(global_mean)

# geo + day mean demand
geo_day_enc = train.groupby(
    ["geohash", "day"]
)["demand"].mean()

train["geo_day_mean_demand"] = (
    train.set_index(["geohash", "day"]).index
    .map(geo_day_enc)
)

test["geo_day_mean_demand"] = (
    test.set_index(["geohash", "day"]).index
    .map(geo_day_enc)
    .fillna(global_mean)
)

# add geo3 smoothed encoding (recommended test: only enable this change first)
enc = smoothed_encoding('geo3', 50)
train['geo3_te'] = train['geo3'].map(enc).fillna(global_mean)
test['geo3_te']  = test['geo3'].map(enc).fillna(global_mean)

features = [
    'geohash', 'geo5', 'geo4', 'geo3', 'RoadType', 'Weather',
    'LargeVehicles', 'Landmarks', 'NumberofLanes',
    'hour', 'minute', 'tmin', 'minute_sin', 'minute_cos', 'hour_sin', 'hour_cos', 'tmin_sin', 'tmin_cos',
    'is_peak', 'is_night', 'time_slot', 'lat', 'lon', 'Temperature',
    'geohash_te', 'geo5_te', 'geo4_te', 'geo3_te', 'geo_day_mean_demand',
]
# include geo3 as a categorical feature for LightGBM and CatBoost
cat_features = ['geohash', 'geo5', 'geo4', 'geo3', 'RoadType', 'Weather',
                'LargeVehicles', 'Landmarks', 'NumberofLanes']
train_df = train.copy()

# daytime window matches the test set; night rows get a smaller weight
daytime = (train_df['tmin'] >= 135) & (train_df['tmin'] <= 825)
weights = np.where(daytime, 1.0, 0.4)

X = train_df[features]
y = train_df['demand']

X_lgb = X.copy()
for c in cat_features:
    X_lgb[c] = X_lgb[c].astype('category')

lgb_model = lgb.LGBMRegressor(
    n_estimators=1500, learning_rate=0.05, num_leaves=128,
    subsample=0.9, colsample_bytree=0.9, random_state=42,
)
lgb_model.fit(X_lgb, y, categorical_feature=cat_features, sample_weight=weights)


# CatBoost predictions
# train a CatBoost model (was missing) and predict with it
cat_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=500,
)

X_cb = X.copy()
for c in cat_features:
    X_cb[c] = X_cb[c].astype(str)

# sanity checks before fit
print('cat_features =', cat_features)
print(X_cb[cat_features].dtypes)

cat_model.fit(
    X_cb,
    y,
    cat_features=cat_features
)

X_test_cb = test[features].copy()
for c in cat_features:
    X_test_cb[c] = X_test_cb[c].astype(str)

pred_cb = cat_model.predict(X_test_cb)

# LightGBM predictions (align categories with the training set)
X_test_lgb = test[features].copy()
for c in cat_features:
    X_test_lgb[c] = pd.Categorical(test[c], categories=X_lgb[c].cat.categories)
pred_lgb = lgb_model.predict(X_test_lgb)

final = np.clip(0.65 * pred_cb + 0.35 * pred_lgb, 0, 1)

submission = pd.DataFrame({'Index': test['Index'].astype(int), 'demand': final})
submission.to_csv('submission2.csv', index=False)
print('saved submission.csv', submission.shape)
submission.head()

train: (77299, 11)  test: (41778, 10)
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006791 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2485
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 27
[LightGBM] [Info] Start training from score 0.099312
cat_features = ['geohash', 'geo5', 'geo4', 'geo3', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks', 'NumberofLanes']
geohash          str
geo5             str
geo4             str
geo3             str
RoadType         str
Weather          str
LargeVehicles    str
Landmarks        str
NumberofLanes    str
dtype: object
0

,Index,demand
0,0,0.056611
1,1,0.013563
2,2,0.076251
3,3,0.008054
4,4,0.068010


In [10]:


import numpy as np
import pandas as pd
import pygeohash as pgh
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# ── 1. LOAD ────────────────────────────────────────────────────────────────────
train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")
print(f"Train: {train.shape}  Test: {test.shape}")

# ── 2. DIAGNOSE DATA STRUCTURE ────────────────────────────────────────────────
print("\n=== DATA DIAGNOSIS ===")
print("Day range train:", train['day'].min(), "→", train['day'].max())
print("Unique days train:", train['day'].nunique())
if 'day' in test.columns:
    print("Day range test:", test['day'].min(), "→", test['day'].max())
    train_days = set(train['day'].unique())
    test_days  = set(test['day'].unique())
    print(f"Test days in train: {len(train_days & test_days)}/{len(test_days)}")

geo_overlap = len(set(train['geohash']) & set(test['geohash']))
print(f"Geohash overlap: {geo_overlap}/{test['geohash'].nunique()} "
      f"({100*geo_overlap/test['geohash'].nunique():.1f}%)")

ts_vals = train['timestamp'].value_counts().head(5)
print("Timestamp sample:\n", ts_vals)

# Detect if Grab dataset (SEA coordinates)
sample_gh = train['geohash'].iloc[0]
lat0, lon0 = pgh.decode(sample_gh)
print(f"\nSample geohash {sample_gh} → lat={lat0:.3f}, lon={lon0:.3f}")
if 1.0 < lat0 < 4.0 and 100 < lon0 < 105:
    print("✓ Likely Grab SEA dataset (Singapore/KL region)")

# ── 3. BASE FEATURE ENGINEERING ───────────────────────────────────────────────
geo_cache = {}
def latlon(g):
    if g not in geo_cache:
        la, lo = pgh.decode(g)
        geo_cache[g] = (float(la), float(lo))
    return geo_cache[g]

def base_features(df):
    df = df.copy()

    # ── Time ──
    hm = df['timestamp'].astype(str).str.split(':', expand=True).astype(int)
    df['hour'], df['minute'] = hm[0], hm[1]
    df['tmin'] = df['hour'] * 60 + df['minute']

    # 96 slots/day (15-min) or 48 (30-min) — auto-detect
    minute_vals = df['minute'].unique()
    slot_divisor = 15 if len(minute_vals) > 2 else 30
    df['time_slot'] = df['hour'] * (60 // slot_divisor) + df['minute'] // slot_divisor

    # Cyclical — tmin is most granular, sufficient
    df['tmin_sin'] = np.sin(2 * np.pi * df['tmin'] / 1440)
    df['tmin_cos'] = np.cos(2 * np.pi * df['tmin'] / 1440)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    # Traffic periods
    df['is_morning_rush'] = ((df['hour'] >= 7) & (df['hour'] <= 9)).astype(int)
    df['is_evening_rush'] = ((df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)
    df['is_peak']  = (df['is_morning_rush'] | df['is_evening_rush']).astype(int)
    df['is_night'] = ((df['hour'] < 6) | (df['hour'] >= 22)).astype(int)
    df['is_weekend'] = (df['day'] % 7 >= 5).astype(int)  # assumes day=0 is Monday

    # Day cyclical (weekly pattern)
    df['day_mod7']     = df['day'] % 7
    df['day_sin']      = np.sin(2 * np.pi * df['day_mod7'] / 7)
    df['day_cos']      = np.cos(2 * np.pi * df['day_mod7'] / 7)

    # ── Geospatial ──
    coords          = df['geohash'].map(latlon)
    df['lat']       = coords.map(lambda t: t[0])
    df['lon']       = coords.map(lambda t: t[1])
    df['geo5']      = df['geohash'].str[:5]
    df['geo4']      = df['geohash'].str[:4]
    # geo3 excluded — zero importance confirmed

    # ── Composite categoricals ──
    df['geo_hour']      = df['geohash'] + '_' + df['hour'].astype(str)
    df['geo_slot']      = df['geohash'] + '_' + df['time_slot'].astype(str)
    df['geo_daymod_slot'] = df['geohash'] + '_' + df['day_mod7'].astype(str) + '_' + df['time_slot'].astype(str)
    df['roadtype_hour'] = df['RoadType'].astype(str) + '_' + df['hour'].astype(str)

    # ── Fill nulls ──
    df['RoadType']    = df['RoadType'].fillna('Unknown')
    df['Weather']     = df['Weather'].fillna('Unknown')

    return df

train = base_features(train)
test  = base_features(test)

temp_med = train['Temperature'].median()
train['Temperature'] = train['Temperature'].fillna(temp_med)
test['Temperature']  = test['Temperature'].fillna(temp_med)

# ── 4. NEIGHBOR DEMAND LOOKUP (uses full train — no leakage, lookup only) ─────
print("\nBuilding neighbor demand lookup...")
geo_mean_all = train.groupby('geohash')['demand'].mean().to_dict()
geo_hour_mean_all = (
    train.groupby(['geohash', 'time_slot'])['demand']
    .mean()
    .to_dict()  # key: (geohash, time_slot)
)

def neighbor_demand(geohash_series, lookup, fallback):
    """Mean demand of 8 spatial neighbors."""
    result = []
    for gh in geohash_series:
        try:
            nbrs = list(pgh.neighbors(gh).values())
            vals = [lookup.get(n, np.nan) for n in nbrs]
            vals = [v for v in vals if not np.isnan(v)]
            result.append(np.mean(vals) if vals else fallback)
        except:
            result.append(fallback)
    return np.array(result)

global_mean = train['demand'].mean()
train['neighbor_demand'] = neighbor_demand(train['geohash'], geo_mean_all, global_mean)
test['neighbor_demand']  = neighbor_demand(test['geohash'],  geo_mean_all, global_mean)

# Neighbor demand at same time slot
def neighbor_demand_slot(df, lookup, fallback):
    result = []
    for gh, slot in zip(df['geohash'], df['time_slot']):
        try:
            nbrs = list(pgh.neighbors(gh).values())
            vals = [lookup.get((n, slot), np.nan) for n in nbrs]
            vals = [v for v in vals if not np.isnan(v)]
            result.append(np.mean(vals) if vals else fallback)
        except:
            result.append(fallback)
    return np.array(result)

train['neighbor_slot_demand'] = neighbor_demand_slot(train, geo_hour_mean_all, global_mean)
test['neighbor_slot_demand']  = neighbor_demand_slot(test,  geo_hour_mean_all, global_mean)

# ── 5. DAY-BASED FORWARD CV SPLIT ─────────────────────────────────────────────
# If test days are future → must use forward splits. If unknown, use this.
# This gives a CV score that correlates much better with LB.

unique_days = np.sort(train['day'].unique())
n_days = len(unique_days)
# Use last 20% of days as validation proxy
val_day_cutoff = unique_days[int(n_days * 0.80)]
print(f"\nDay-based split: train days ≤ {val_day_cutoff}, val days > {val_day_cutoff}")
print(f"Train rows: {(train['day'] <= val_day_cutoff).sum()}, "
      f"Val rows: {(train['day'] > val_day_cutoff).sum()}")

# For full CV we use 5 forward folds
def day_forward_splits(train_df, n_splits=5):
    """
    Forward-walk day splits. Each fold: train on first k chunks, val on next chunk.
    Simulates predicting future days — matches LB evaluation.
    """
    days = np.sort(train_df['day'].unique())
    # divide days into n_splits+1 equal chunks
    chunks = np.array_split(days, n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_days = np.concatenate(chunks[:i])
        val_days   = chunks[i]
        tr_idx  = train_df.index[train_df['day'].isin(train_days)].tolist()
        val_idx = train_df.index[train_df['day'].isin(val_days)].tolist()
        splits.append((tr_idx, val_idx))
    return splits

splits = day_forward_splits(train, n_splits=5)

# ── 6. FOLD-SAFE TARGET ENCODING FUNCTION ────────────────────────────────────
def compute_target_encodings(tr_df, val_df, te_df, target_col='demand'):
    """
    Compute all demand-based aggregates on tr_df only.
    Apply (map) to val_df and te_df — zero leakage.
    Returns three DataFrames of new columns.
    """
    gm = tr_df[target_col].mean()  # global mean of this fold's train

    def agg_encode(group_keys, stat='mean', smooth=None, count_smooth=20):
        """Smoothed mean encoding: pulls toward global mean for rare groups."""
        g = tr_df.groupby(group_keys)[target_col]
        stats = g.agg(['mean', 'count'])
        if smooth:
            stats['enc'] = (
                (stats['mean'] * stats['count'] + gm * count_smooth) /
                (stats['count'] + count_smooth)
            )
        else:
            stats['enc'] = stats[stat]
        return stats['enc']

    results = {}

    # ── Geohash-level ──
    for col in ['geohash', 'geo5', 'geo4']:
        smooth_k = {'geohash': 15, 'geo5': 25, 'geo4': 40}[col]
        enc = agg_encode(col, smooth=True, count_smooth=smooth_k)
        results[f'{col}_mean_demand'] = enc

    # ── geo × time_slot (most powerful) ──
    enc_gh_slot = agg_encode(['geohash', 'time_slot'], smooth=True, count_smooth=5)
    results['geo_slot_mean'] = enc_gh_slot

    # ── geo × hour ──
    enc_gh_hour = agg_encode(['geohash', 'hour'], smooth=True, count_smooth=5)
    results['geo_hour_mean'] = enc_gh_hour

    # ── geo × day_mod7 × time_slot (captures weekly patterns per location) ──
    enc_gh_day_slot = agg_encode(['geohash', 'day_mod7', 'time_slot'],
                                  smooth=True, count_smooth=3)
    results['geo_day_slot_mean'] = enc_gh_day_slot

    # ── RoadType × time_slot ──
    enc_rt_slot = agg_encode(['RoadType', 'time_slot'], smooth=True, count_smooth=10)
    results['roadtype_slot_mean'] = enc_rt_slot

    # ── geo4 × time_slot ──
    enc_geo4_slot = agg_encode(['geo4', 'time_slot'], smooth=True, count_smooth=10)
    results['geo4_slot_mean'] = enc_geo4_slot

    # ── geo × is_peak ──
    enc_gh_peak = agg_encode(['geohash', 'is_peak'], smooth=True, count_smooth=5)
    results['geo_peak_mean'] = enc_gh_peak

    # ── Demand variability (std / mean = CoV) — noise robustness ──
    geo_std  = tr_df.groupby('geohash')[target_col].std().fillna(0)
    geo_mean_vals = tr_df.groupby('geohash')[target_col].mean()
    geo_cov  = (geo_std / (geo_mean_vals + 1e-6)).rename('geo_demand_cov')
    results['geo_demand_cov'] = geo_cov

    # ── Demand rank within geo4 parent (distribution-shift robust) ──
    geo_mean_df = geo_mean_vals.reset_index()
    geo_mean_df.columns = ['geohash', 'geo_mean_v']
    geo_mean_df['geo4'] = geo_mean_df['geohash'].str[:4]
    geo_mean_df['geo4_rank'] = geo_mean_df.groupby('geo4')['geo_mean_v'].rank(pct=True)
    geo4_rank_lookup = geo_mean_df.set_index('geohash')['geo4_rank']

    # ── Apply to all three splits ──
    def apply_encodings(df):
        out = pd.DataFrame(index=df.index)

        def safe_map(df_in, keys, enc_series, col_name):
            """Works for single or multi-key encodings."""
            if isinstance(keys, str):
                mapped = df_in[keys].map(enc_series)
            else:
                mapped = df_in.set_index(keys).index.map(
                    enc_series.to_dict().get
                )
                mapped = pd.Series(mapped, index=df_in.index)
            out[col_name] = mapped.fillna(gm)

        safe_map(df, 'geohash',                  results['geohash_mean_demand'],   'geohash_mean_demand')
        safe_map(df, 'geo5',                     results['geo5_mean_demand'],      'geo5_mean_demand')
        safe_map(df, 'geo4',                     results['geo4_mean_demand'],      'geo4_mean_demand')
        safe_map(df, ['geohash', 'time_slot'],   results['geo_slot_mean'],         'geo_slot_mean')
        safe_map(df, ['geohash', 'hour'],        results['geo_hour_mean'],         'geo_hour_mean')
        safe_map(df, ['geohash', 'day_mod7', 'time_slot'], results['geo_day_slot_mean'], 'geo_day_slot_mean')
        safe_map(df, ['RoadType', 'time_slot'],  results['roadtype_slot_mean'],    'roadtype_slot_mean')
        safe_map(df, ['geo4', 'time_slot'],      results['geo4_slot_mean'],        'geo4_slot_mean')
        safe_map(df, ['geohash', 'is_peak'],     results['geo_peak_mean'],         'geo_peak_mean')
        out['geo_demand_cov']  = df['geohash'].map(geo_cov).fillna(0)
        out['geo4_demand_rank'] = df['geohash'].map(geo4_rank_lookup).fillna(0.5)

        return out

    te_tr  = apply_encodings(tr_df)
    te_val = apply_encodings(val_df)
    te_te  = apply_encodings(te_df)

    return te_tr, te_val, te_te

# ── 7. DEFINE BASE FEATURES ───────────────────────────────────────────────────
BASE_FEATURES = [
    'hour', 'minute', 'tmin', 'time_slot', 'day', 'day_mod7',
    'tmin_sin', 'tmin_cos', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'is_morning_rush', 'is_evening_rush', 'is_peak', 'is_night', 'is_weekend',
    'lat', 'lon',
    'NumberofLanes', 'Temperature',
    'neighbor_demand', 'neighbor_slot_demand',
    'geohash', 'geo5', 'geo4',
    'RoadType', 'Weather', 'LargeVehicles', 'Landmarks',
    'geo_hour', 'geo_slot', 'roadtype_hour',
]

# Target encoding columns added inside CV
TE_COLS = [
    'geohash_mean_demand', 'geo5_mean_demand', 'geo4_mean_demand',
    'geo_slot_mean', 'geo_hour_mean', 'geo_day_slot_mean',
    'roadtype_slot_mean', 'geo4_slot_mean', 'geo_peak_mean',
    'geo_demand_cov', 'geo4_demand_rank',
]

ALL_FEATURES = BASE_FEATURES + TE_COLS

CAT_FEATURES_CB = [
    'geohash', 'geo5', 'geo4',
    'RoadType', 'Weather', 'LargeVehicles', 'Landmarks',
    'geo_hour', 'geo_slot', 'roadtype_hour',
]
# NumberofLanes kept numeric — CatBoost treats it better as int

# ── 8. CV LOOP — CatBoost ─────────────────────────────────────────────────────
print("\n=== CatBoost CV (day-forward splits) ===")
X_all   = train[BASE_FEATURES].copy()
y_all   = train['demand'].copy()
X_test_ = test[BASE_FEATURES].copy()

oof_cb   = np.zeros(len(train))
test_cb  = np.zeros(len(test))
cb_scores = []

cb_params = dict(
    iterations=4000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    min_data_in_leaf=15,
    random_strength=0.5,
    bagging_temperature=0.8,
    od_type='Iter',
    od_wait=300,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=500,
)

for fold, (tr_idx, val_idx) in enumerate(splits, 1):
    X_tr   = train.loc[tr_idx, BASE_FEATURES].copy()
    X_val  = train.loc[val_idx, BASE_FEATURES].copy()
    y_tr   = train.loc[tr_idx, 'demand']
    y_val  = train.loc[val_idx, 'demand']

    # Fold-safe target encodings
    te_tr, te_val, te_te = compute_target_encodings(
        train.loc[tr_idx], train.loc[val_idx], test
    )

    X_tr_full  = pd.concat([X_tr.reset_index(drop=True),
                             te_tr.reset_index(drop=True)], axis=1)
    X_val_full = pd.concat([X_val.reset_index(drop=True),
                             te_val.reset_index(drop=True)], axis=1)
    X_te_full  = pd.concat([X_test_.reset_index(drop=True),
                             te_te.reset_index(drop=True)], axis=1)

    # Sample weights: upweight daytime rows to match test distribution
    daytime = (train.loc[tr_idx, 'tmin'] >= 135) & (train.loc[tr_idx, 'tmin'] <= 825)
    w = np.where(daytime.values, 1.0, 0.5)

    model = CatBoostRegressor(**cb_params)
    model.fit(
        X_tr_full, y_tr,
        cat_features=CAT_FEATURES_CB,
        eval_set=(X_val_full, y_val),
        use_best_model=True,
        sample_weight=w,
        verbose=500,
    )

    oof_cb[val_idx] = model.predict(X_val_full)
    test_cb        += model.predict(X_te_full) / len(splits)

    score = r2_score(y_val, oof_cb[val_idx])
    cb_scores.append(score)
    print(f"  Fold {fold}: R² = {score:.5f}")

oof_r2_cb = r2_score(y_all, oof_cb)
print(f"\nCatBoost OOF R² = {oof_r2_cb:.5f}  (mean fold = {np.mean(cb_scores):.5f})")
print("NOTE: OOF R² is the honest estimate. Mean-fold R² is optimistic for forward splits.")

# ── 9. LIGHTGBM CV LOOP ───────────────────────────────────────────────────────
print("\n=== LightGBM CV (day-forward splits) ===")
oof_lgb   = np.zeros(len(train))
test_lgb  = np.zeros(len(test))
lgb_scores = []

lgb_params = dict(
    n_estimators=4000,
    learning_rate=0.03,
    num_leaves=127,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.75,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

CAT_FEATURES_LGB = [c for c in CAT_FEATURES_CB
                    if c in ALL_FEATURES]

for fold, (tr_idx, val_idx) in enumerate(splits, 1):
    X_tr   = train.loc[tr_idx, BASE_FEATURES].copy()
    X_val  = train.loc[val_idx, BASE_FEATURES].copy()
    y_tr   = train.loc[tr_idx, 'demand']
    y_val  = train.loc[val_idx, 'demand']

    te_tr, te_val, te_te = compute_target_encodings(
        train.loc[tr_idx], train.loc[val_idx], test
    )

    X_tr_full  = pd.concat([X_tr.reset_index(drop=True),
                             te_tr.reset_index(drop=True)], axis=1)
    X_val_full = pd.concat([X_val.reset_index(drop=True),
                             te_val.reset_index(drop=True)], axis=1)
    X_te_full  = pd.concat([X_test_.reset_index(drop=True),
                             te_te.reset_index(drop=True)], axis=1)

    # Label encode categoricals for LGBM
    from sklearn.preprocessing import LabelEncoder
    X_tr_lgb  = X_tr_full.copy()
    X_val_lgb = X_val_full.copy()
    X_te_lgb  = X_te_full.copy()
    le_dict   = {}
    for c in CAT_FEATURES_LGB:
        le = LabelEncoder()
        combined = pd.concat([X_tr_lgb[c], X_val_lgb[c], X_te_lgb[c]]).astype(str)
        le.fit(combined)
        X_tr_lgb[c]  = le.transform(X_tr_lgb[c].astype(str))
        X_val_lgb[c] = le.transform(X_val_lgb[c].astype(str))
        X_te_lgb[c]  = le.transform(X_te_lgb[c].astype(str))
        le_dict[c]   = le

    daytime = (train.loc[tr_idx, 'tmin'] >= 135) & (train.loc[tr_idx, 'tmin'] <= 825)
    w = np.where(daytime.values, 1.0, 0.5)

    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(
        X_tr_lgb, y_tr,
        sample_weight=w,
        eval_set=[(X_val_lgb, y_val)],
        callbacks=[lgb.early_stopping(300, verbose=False),
                   lgb.log_evaluation(500)],
        categorical_feature=CAT_FEATURES_LGB,
    )

    oof_lgb[val_idx] = model_lgb.predict(X_val_lgb)
    test_lgb        += model_lgb.predict(X_te_lgb) / len(splits)

    score = r2_score(y_val, oof_lgb[val_idx])
    lgb_scores.append(score)
    print(f"  Fold {fold}: R² = {score:.5f}")

oof_r2_lgb = r2_score(y_all, oof_lgb)
print(f"\nLightGBM OOF R² = {oof_r2_lgb:.5f}  (mean fold = {np.mean(lgb_scores):.5f})")

# ── 10. OPTIMAL BLEND ─────────────────────────────────────────────────────────
from scipy.optimize import minimize_scalar

def neg_blend_r2(w):
    blended = w * oof_cb + (1 - w) * oof_lgb
    return -r2_score(y_all, blended)

res = minimize_scalar(neg_blend_r2, bounds=(0.0, 1.0), method='bounded')
best_w = res.x
blend_r2 = r2_score(y_all, best_w * oof_cb + (1 - best_w) * oof_lgb)
print(f"\nOptimal blend: CatBoost={best_w:.3f}, LightGBM={1-best_w:.3f}")
print(f"Blended OOF R² = {blend_r2:.5f}")

final_preds = best_w * test_cb + (1 - best_w) * test_lgb
final_preds = np.clip(final_preds, 0, 1)

# ── 11. FEATURE IMPORTANCE ────────────────────────────────────────────────────
# Use last fold's CatBoost model
imp_df = pd.DataFrame({
    'Feature':    model.feature_names_,
    'Importance': model.get_feature_importance(),
}).sort_values('Importance', ascending=False)
print("\n=== Feature Importance (last CB fold) ===")
print(imp_df.head(25).to_string(index=False))

# ── 12. SUBMISSION ────────────────────────────────────────────────────────────
submission = pd.DataFrame({
    'Index':  test['Index'].astype(int),
    'demand': final_preds,
})
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission saved: {submission.shape}")
print(submission.head())
print(f"\nDemand stats: min={final_preds.min():.4f}, "
      f"max={final_preds.max():.4f}, mean={final_preds.mean():.4f}")

# ── 13. CV SUMMARY ───────────────────────────────────────────────────────────
print("\n=== FINAL SUMMARY ===")
print(f"CatBoost  OOF R²: {oof_r2_cb:.5f}")
print(f"LightGBM  OOF R²: {oof_r2_lgb:.5f}")
print(f"Ensemble  OOF R²: {blend_r2:.5f}")
print(f"CB weight: {best_w:.3f} | LGB weight: {1-best_w:.3f}")
print("\nIf this CV score is ~0.91–0.94, expect LB ≈ 91–94.")
print("If CV >> LB still, suspect test covers unseen geohashes or future dates beyond training range.")

Train: (77299, 11)  Test: (41778, 10)

=== DATA DIAGNOSIS ===
Day range train: 48 → 49
Unique days train: 2
Day range test: 49 → 49
Test days in train: 1/1
Geohash overlap: 1180/1190 (99.2%)
Timestamp sample:
 timestamp
2:0     1778
1:45    1755
1:30    1750
1:15    1698
1:0     1668
Name: count, dtype: int64

Sample geohash qp02z1 → lat=-5.485, lon=90.665

Building neighbor demand lookup...

Day-based split: train days ≤ 49, val days > 49
Train rows: 77299, Val rows: 0

=== CatBoost CV (day-forward splits) ===


CatBoostError: Invalid type for cat_feature[non-default value idx=0,feature_idx=32]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.